## `APPLY CHANGES INTO` — declarative CDC & SCD

Hand-writing `MERGE INTO` for change data capture works (module 02) but it's verbose and race-prone. **`APPLY CHANGES INTO`** is the declarative-pipeline equivalent — CDC with SCD type 1 or type 2 baked in.

```sql
CREATE OR REFRESH STREAMING TABLE silver_customers;

APPLY CHANGES INTO live.silver_customers
FROM   STREAM(live.bronze_customers_cdc)
KEYS   (customer_id)
APPLY AS DELETE     WHEN _change_type = 'delete'
SEQUENCE BY _commit_version
COLUMNS    * EXCEPT (_change_type, _commit_version)
STORED AS SCD TYPE 1;     -- or SCD TYPE 2 for history
```

**Five clauses worth memorising:**

- **`KEYS (...)`** — the natural key for the upsert.
- **`SEQUENCE BY ...`** — the ordering column that decides "latest wins" for late-arriving changes.
- **`APPLY AS DELETE WHEN ...`** — propagate hard deletes from the source.
- **`STORED AS SCD TYPE 1`** — overwrite in place, no history.
- **`STORED AS SCD TYPE 2`** — keep history with start/end timestamps and an is-current flag, all maintained for you.

**Why the exam loves it:** a hand-rolled SCD type 2 `MERGE` is a multi-statement, race-prone affair. `APPLY CHANGES INTO ... STORED AS SCD TYPE 2` is one declarative block that does it all correctly. When a question asks for "declarative CDC keeping full history," this is the answer — not `MERGE`.
